# 04 — Speaking: `monad_speak` / `monad_speak_wick`, stage by stage
C source: `monad.c` lines 560–1045. Port: `Monad.speak`, `Monad.speak_wick`, `near_canonical`, `fermat_space`.
`-W` = `monad_speak_wick` = `monad_speak` with `affect` forced to −1, so the emission gate is `cos(γ/2 − π/2) = sin(γ/2) > 0`.

In [1]:
import os, sys, json, math, collections, re, hashlib
import numpy as np
sys.path.insert(0, os.path.abspath("../tools"))
import ptol_bin, monad_py as mp
ROOT  = os.path.abspath("..")
STORE = os.path.join(ROOT, "dirty", "monad_wordnet_v4_20260519.bin")
CBIN  = os.path.join(ROOT, "build", "PtolC_eb28ce8", "ptolemy")
SPACES = "[\u200b\u2009\u2008\u2007\u2006\u2005\u2004\u2003]"   # the eight Fermat separators

## Run the screenshot prompt and expose every stage

In [2]:
m = mp.Monad.from_bin(STORE); tr = {}
out = m.speak_wick("what is your name", 50, tr)
print(repr(out)); print(re.split(SPACES, out))

'holcus\u200bsame\u200bstuff\u200bseemy'
['holcus', 'same', 'stuff', 'seemy']


### Stage 1 — hear
Each token is addressed (`monad_wm_get`). A word the store already displays keeps its stored `E`; any other word is computed, and flagged `[new]` in the C trace.

In [3]:
for w, idx, E, known in tr["hear"]:
    print(f"{w:6} z#{idx:<6} gamma={m.zeros[idx]:10.3f}  E={E:.4f}  beta={m.beta[idx]:.3f}  known={known}   slot displays {m.vocab.get(idx,{}).get('word')!r}")

what   z#16194  gamma= 15010.512  E=0.4540  beta=7.552  known=True   slot displays 'what'
is     z#1894   gamma=  2405.433  E=0.2703  beta=7.552  known=False   slot displays 'nanti'
your   z#18893  gamma= 17171.526  E=0.4887  beta=7.552  known=True   slot displays 'your'
name   z#15471  gamma= 14425.021  E=0.4447  beta=7.552  known=False   slot displays 'emite'


### Stage 2 — seed, spread, two A-passes
`J[idx] += β·E²·exp(−λ·age)`; each seeded zero then leaks `J·exp(−|Δn|)` to ±3 neighbours; then every A edge is crossed twice from the *pre-propagation* field `J0`
(`i→j` weighted by `β[j]`, then `j→i` weighted by `β[i]`), edge weight clamped to `1/gap`.

In [4]:
print("lag_ratio", tr["lag_ratio"], " sum_V", f'{tr["sum_V"]:.4e}', " sum_T", f'{tr["sum_T"]:.4f}', " -> actual_max", tr["actual_max"])
print("J_max", f'{tr["J_max"]:.3f}', " J_floor (1% of J_max)", f'{tr["J_floor"]:.3f}')

lag_ratio 0.00012627387474987232  sum_V 5.6683e+03  sum_T 0.7158  -> actual_max 4
J_max 988.952  J_floor (1% of J_max) 9.890


### Stage 3 — length from a Lagrangian ratio
`actual_max = 4 + int(max_tokens · ΣT/(ΣT+ΣV))`, `ΣV = Σ β·E²` over every occupied slot (≈ 5.7·10³ here), `ΣT = Σ E²` over the query's slots (< 1).
The ratio is ≈ 10⁻⁴, so `actual_max = 4` for any short prompt on this store.

### Stage 4 — Fermat pointer and the golden-angle step

In [5]:
print("pointer z#", tr["pointer"], " gamma", round(m.zeros[tr["pointer"]],3), "(median gamma of the query's zeros, nearest zero)")
print("golden step = round(N/phi^2) =", tr["golden_step"])

pointer z# 16194  gamma 15010.512 (median gamma of the query's zeros, nearest zero)
golden step = round(N/phi^2) = 9549


### Stage 5 — first word ("demotic") and the walk

In [6]:
print("target_E = (sumV/(sumV+sumT))*Omega =", tr["target_E"], "  chosen slot", tr["demotic"], "word", m.vocab[tr["demotic"]]["word"], "E", m.vocab[tr["demotic"]]["E"])
for e in tr["emitted"]: print(e)

target_E = (sumV/(sumV+sumT))*Omega = 0.5670683850346744   chosen slot 24639 word holcus E 0.5625142041015625
('demotic', 24639, 'holcus', 'holcus')
('walk', 1859, 'same', 'same', 47.90879803589318, '\u200b')
('walk', 18482, 'stuff', 'stuff', 10.98523437021878, '\u200b')
('walk', 0, 'seemy', 'seemy', 104.77826071394702, '\u200b')


`('walk', slot, stored word, displayed word, J, separator)`. `near_canonical` may replace the stored word by a neighbour within ±50 slots (`same`, `stuff` here were the slots' own words).
The separator is `U+200B … U+2003` chosen by `J/J_max` in eight bands — the "Fermat space". The sequence `holcus​same​stuff​seemy` carries zero-width spaces: the screenshot shows no spaces for that reason.

## Equivalence with the compiled binary (8 prompts, byte for byte)

In [7]:
import verify_against_c as vc
ok = 0
for p in ["what is your name","can you hear what i am typing","who are you","what is a tree","good morning","the cat sat on the mat","how do you do","tell me about water"]:
    c = vc.c_reply(p); mm = mp.Monad.from_bin(STORE); py = mm.speak_wick(p, 50); ok += (c == py)
    print("MATCH" if c == py else "DIFF ", repr(p), "->", re.split(SPACES, c))
print(ok, "/ 8 identical")

MATCH 'what is your name' -> ['holcus', 'same', 'stuff', 'seemy']


MATCH 'can you hear what i am typing' -> ['holcus', 'setn', 'abysmal', 'quun']


MATCH 'who are you' -> ['holcus', 'abysmal', 'quun', 'tuxen']


MATCH 'what is a tree' -> ['holcus', 'jome', 'logzh', 'cold']


MATCH 'good morning' -> ['holcus', 'conoy', 'nland', 'abusive']


MATCH 'the cat sat on the mat' -> ['holcus', 'bacoo', 'seemy', 'nanti']


MATCH 'how do you do' -> ['holcus', 'behim', 'celiac', 'stuff']


MATCH 'tell me about water' -> ['holcus', 'piezo', 'nazri', 'geolb']
8 / 8 identical
